# 03.01 — Cypher Query Definitions

Orthograph provides a concrete, YAML-serialisable way to define a Cypher query:

1. **`CypherQuery`** — a concrete, YAML-serialisable Pydantic data class you instantiate directly.
   Useful for runtime query construction, YAML-loaded catalogues, or one-off definitions.

This notebook covers:
- Instantiating `CypherQuery` with parameters and optional typing
- Calling `.build()` to produce `CypherQueryData` (cypher string template + params dict)
- Loading a YAML catalogue with `load_query_catalogue`
- Validating query definitions against a `GraphDefinition` (static, no database)
- Interpreting raw Cypher driver results

In [ ]:
from pydantic import BaseModel

from orthograph.api.model import load_query_catalogue, validate_query
from orthograph.cypher.bindings import NoIdentifiers
from orthograph.cypher.query import CypherQuery
from orthograph.graph_definition.graph_definition import GraphDefinition

## 1. Define the domain model

We reuse the filmography domain: `Person`, `Movie`, and `ActedIn`.

In [ ]:
from shared.filmography import ActedIn, Movie, Person


# Create a GraphDefinition for validation
graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

print("Domain model loaded:")
print(f"  Nodes: {[nt.__label__ for nt in graph_definition.node_types]}")
print(
    f"  Relationships: {[rt.__label__ for rt in graph_definition.relationship_types]}"
)

## 2. CypherQuery instantiation

`CypherQuery` is a concrete, YAML-serialisable Pydantic model you instantiate directly.
Declare a `Params` model to type the `$param` names your Cypher uses.
For zero-arg queries, pass `Params=NoParams`.

In [ ]:
# Simple query with a params_schema model — required field title.
class FindMovieByTitleParams(BaseModel):
    title: str


simple_query = CypherQuery(
    query_id="find_movie_by_title",
    cypher_template="MATCH (m:Movie {title: $title}) RETURN m.title, m.released",
    params_schema=FindMovieByTitleParams,
    description="Find a movie by exact title match",
    identifiers_schema=NoIdentifiers,
)

print(f"Query id: {simple_query.query_id}")
print(f"Arguments: {simple_query.list_arguments()}")
print(f"Description: {simple_query.description}")

## 3. CypherQuery with a Params model

Attach a `Params` Pydantic model to enable type validation and coercion at build time.

In [ ]:
class MoviesByYearParams(BaseModel):
    released: int
    limit: int = 10


cypher_query = CypherQuery(
    query_id="movies_by_year",
    cypher_template=(
        "MATCH (m:Movie {released: $released}) RETURN m.title, m.released LIMIT $limit"
    ),
    params_schema=MoviesByYearParams,
    description="Find movies released in a given year",
    identifiers_schema=NoIdentifiers,
)

print(f"Query: {cypher_query.query_id}")
print(f"params_schema fields: {list(cypher_query.params_schema.model_fields.keys())}")
print(f"Argument lists: {cypher_query.list_arguments()}")

## 4. Building queries with .build()

Call `.build(**kwargs)` to produce a `CypherQueryData` (a NamedTuple with `cypher` and `params`).
The returned value is ready to pass directly to a driver session.

In [ ]:
# Build with just required args
query_data = simple_query.build(title="The Matrix")
print("Simple query result:")
print(f"  Cypher: {query_data.cypher}")
print(f"  Params: {query_data.params}")
print()

# Unpack the NamedTuple
cypher, params = query_data
print(f"Unpacked: cypher={cypher}")
print(f"Unpacked: params={params}")

## 5. CypherQueryData unpacking

`CypherQueryData` is a `NamedTuple`, so it unpacks naturally: `cypher, params = query.build(...)`

In [ ]:
# Build a typed query with optional arg overridden
cypher, params = cypher_query.build(released=1999, limit=5)
print(f"Cypher:\n  {cypher}")
print(f"\nParams: {params}")
print(f"\nParams type: {type(params)}")
print(f"Params keys: {list(params.keys())}")

## 6. Type validation at build time

When a `Params` model is attached, `.build()` validates and coerces argument values through it.

In [ ]:
# Coerce string "1999" to int
cypher, params = cypher_query.build(released="1999")
print("Passed released='1999' (string)")
print(f"Params: {params}")
print(f"Type of released in params: {type(params['released'])}")

## 7. Validating query definitions

Use `validate_query(definition)` to check a `CypherQuery` definition against a `GraphDefinition`.
This runs static checks (no database) for unknown labels, relationship types, and properties.
Unknown names surface as ERRORs.

In [ ]:
# Validate a valid query
result = validate_query(simple_query.cypher_template, graph_definition)
print(f"Validation result: is_valid={result.is_valid}")
print(f"Issues: {len(result.issues)}")
for issue in result.issues:
    print(f"  {issue}")

## 9. JSON serialization

`CypherQuery` is fully serializable. Use `.model_dump(by_alias=True, exclude_none=True)`
to get the wire format: `query_id`, `params_schema` for the params_schema
JSON Schema, and `identifiers_schema` for identifiers_schema.

In [ ]:
# Query with unknown label
class BadLabelParams(BaseModel):
    label: str


bad_query = CypherQuery(
    query_id="bad_label",
    cypher_template="MATCH (s:Studio {label: $label}) RETURN s.label",
    params_schema=BadLabelParams,
    identifiers_schema=NoIdentifiers,
)

result = validate_query(bad_query.cypher_template, graph_definition)
print(f"Validation result: is_valid={result.is_valid}")
print(f"Issues: {len(result.issues)}")
for issue in result.issues:
    print(f"  {issue}")

## 9. JSON serialization

`CypherQuery` is fully serializable. Use `.model_dump(by_alias=True, exclude_none=True)`
to get the wire format: `query_name` alias for `name`, `params_schema` for the Params
JSON Schema, and `identifiers_schema` for Identifiers.

In [ ]:
import json


# Serialize with aliases (for JSON)
serialized = simple_query.model_dump(by_alias=True, exclude_none=True)
print("Serialized (with aliases for JSON):")
print(json.dumps(serialized, indent=2))

## 10. Loading a  catalogue

Use `load_query_catalogue(source)` to load a list of `CypherQuery` instances from a YAML file or string.
Each entry in the YAML list becomes a `CypherQuery`.

In [ ]:
# Create a small YAML catalogue as a string
yaml_content = """
- query_id: find_movie
  cypher_template: "MATCH (m:Movie {title: $title}) RETURN m.title, m.released"
  description: "Find a movie by title"
  params_schema:
    title: FindMovieParams
    type: object
    properties:
      title: {type: string, title: Title}
    required: [title]

- query_id: actor_by_name
  cypher_template: "MATCH (p:Person {name: $name}) RETURN p.name, p.born"
  description: "Find a person by name"
  params_schema:
    title: ActorByNameParams
    type: object
    properties:
      name: {type: string, title: Name}
    required: [name]
"""

# Load the catalogue
queries = load_query_catalogue(yaml_content)
print(f"Loaded {len(queries)} queries:")
for q in queries:
    print(f"  - {q.query_id}: {q.description}")

## 11. Building queries from the catalogue

After loading, build each query just like any other `CypherQuery`.

In [ ]:
# Use the first query
movie_query = queries[0]
cypher, params = movie_query.build(title="Fight Club")
print(f"Query: {movie_query.query_id}")
print(f"Cypher: {cypher}")
print(f"Params: {params}")
print()

# Use the second query
person_query = queries[1]
cypher, params = person_query.build(name="Tom Hanks")
print(f"Query: {person_query.query_id}")
print(f"Cypher: {cypher}")
print(f"Params: {params}")

## 12. Raw Cypher result unpacking

`CypherQuery` returns only `(cypher, params)` — no output model or materialization.
Callers receive raw driver results as dicts and unpack them themselves.

In [ ]:
# Simulate raw results from a driver (list of dicts)
raw_results = [
    {"m.title": "The Matrix", "m.released": 1999},
    {"m.title": "Speed", "m.released": 1994},
]

# When you have real results, unpack them manually
for raw in raw_results:
    title = raw["m.title"]
    released = raw["m.released"]
    print(f"Movie: {title} ({released})")

## 13. Combining hand-built and YAML queries

You can freely mix hand-built `CypherQuery` instances with catalogue queries loaded from YAML —
they are all `CypherQuery` instances and handled uniformly.

In [ ]:
# Hand-built query
class FindMovieByYearParams(BaseModel):
    year: int


class_query = CypherQuery(
    query_id="find_movie_by_year",
    cypher_template="MATCH (m:Movie {released: $year}) RETURN m.title, m.released",
    params_schema=FindMovieByYearParams,
    identifiers_schema=NoIdentifiers,
)

# Load from YAML
yaml_queries = load_query_catalogue(yaml_content)

# All are CypherQuery instances
all_queries = [class_query] + yaml_queries
print(f"Combined query list ({len(all_queries)} total):")
for q in all_queries:
    print(f"  - {q.query_id}")

## 14. Summary

| Aspect | CypherQuery |
|--------|-------------|
| **Usage** | Direct instantiation (data class) or YAML loading |
| **Validation** | At construction time and via `validate_query` (static, no DB) |
| **YAML** | Directly serializable and loadable |
| **When to use** | One-off queries, YAML catalogues, runtime construction |

**Key points:**
- `.build()` returns `CypherQueryData(cypher, params)` — ready for a driver session.
- `Params` is **required** — pass `NoParams` for zero-arg queries.
- Use `validate_query(cypher_template, graph_definition)` for static schema checks (no database).
- Raw results from a driver are dicts — callers unpack them manually (no built-in materialization).
- Load YAML catalogues with `load_query_catalogue(source)` — each entry becomes a `CypherQuery`.
- YAML format uses `cypher_template` + `params_schema` (JSON Schema object).